In [ ]:
%run ./PY_INITIALIZE.ipynb

In [ ]:
import numpy as np
import modin.pandas as pd
import snowflake.snowpark.modin.plugin

def calculate_psi(baseline, current,dtype_dict, num_buckets=10):
    """Calculates PSI between training (baseline) and production (current) features."""
    df_baseline = pd.read_snowflake(f"{database_name}.{schema_name}.{baseline}")
    df_current = pd.read_snowflake(f"{database_name}.{schema_name}.{current}")
    df_baseline.fillna(0, inplace=True)
    df_current.fillna(0, inplace=True)
    df_baseline.astype(dtype_dict)
    df_current.astype(dtype_dict)
    """list features and save it"""
    features = list(dtype_dict.keys())
    """Create a dataframe with features as column headers
    psi_values = pd.DataFrame(columns=features)"""
    psi_values = np.zeros(len(features))
    dtypes = list(dtype_dict.values())
    # Determine bucket thresholds based on training data
    for i in range(len(dtype_dict)):
        feature = features[i]
        print(feature)
        baseline_feature = (df_baseline[feature]).to_numpy().flatten()
        current_feature = (df_current[feature]).to_numpy().flatten()
        percentiles = np.linspace(0, 100, num_buckets + 1)
        baseline_feature = baseline_feature.astype(dtypes[i])
        current_feature = current_feature.astype(dtypes[i])
        buckets = np.percentile(baseline_feature, percentiles)
        buckets[0], buckets[-1] = -np.inf, np.inf
        
        # Calculate counts in each bucket
        baseline_counts, _ = np.histogram(baseline_feature, bins=buckets)
        current_counts, _ = np.histogram(current_feature, bins=buckets)
        
        # Convert to fractions with smoothing to prevent division by zero
        baseline_pct = np.where(baseline_counts == 0, 0.0001, baseline_counts) / len(baseline_feature)
        current_pct = np.where(current_counts == 0, 0.0001, current_counts) / len(current_feature)
        
        # Calculate total PSI
        psi_value = np.sum((current_pct - baseline_pct) * np.log(current_pct / baseline_pct))
        """ After calculation psi value for the training feature, the 
        psi values  dataframe is updated for each feature"""
        #psi_values.iat[0, psi_values.columns.get_loc(feature)] = psi_value
        psi_values[i]= psi_value
    return psi_values

# Usage: Trigger alert if calculate_psi(X_train['feature'], X_production['feature']) > 0.25


In [ ]:
feature_dtype_dict = {'PT_AGE':int,'PT_ZIP':float,'ICDCD_NUMCODED':float, \
    'CLM_DIS_RISK_NBR':int,'SUBCD_NBR':float}

In [ ]:

#feature_psi = pd.DataFrame(columns=['PT_AGE', 'PT_ZIP','ICDCD_NUMCODED','CLM_DIS_RISK_NBR','SUBCD_NBR'])
feature_psi_values = calculate_psi("TRAINING_DATA", "TESTING_DATA", feature_dtype_dict, 10)


In [ ]:
TotalFeatures = len(feature_dtype_dict)
print(feature_psi_values)
feature_psi_2D = feature_psi_values.reshape(-1, 5)
feature_psi = pd.DataFrame(feature_psi_2D, columns=list(feature_dtype_dict.keys()))


In [ ]:
import pandas as pd
from snowflake.snowpark import Session
feature_psi.to_snowflake(
            name=f"TrainingFeaturesPsi", 
            index=False,  # Automatically creates table if missing
            if_exists= 'replace'           # Overwrites contents if table exists
        )
#psi_snowpark = session.create_dataframe(psi_pandas)
#psi_snowpark.write.mode("overwrite").save_as_table(f"{database_name}.{schema_name}.TrainingFeaturesPsi")